# บทที่ 5 — สร้างและเทรนโมเดลแรก (LSTM)

<sub>บทเรียนที่ 5 จาก 8 &nbsp;·&nbsp; [← บทที่ 4](04_data_preparation.ipynb) · [สารบัญ](README.md) · [บทที่ 6 →](06_comparing_architectures.ipynb)</sub>

## เป้าหมายของบทนี้

เมื่อจบบทนี้คุณจะ:

- เข้าใจว่า LSTM ทำงานอย่างไรและทำไมเหมาะกับข้อมูลลำดับเวลา
- เขียนโมเดล LSTM ด้วย PyTorch ตั้งแต่ต้น
- เทรนโมเดลจริงและอ่านกราฟ loss เป็น
- รู้จัก early stopping และเหตุผลที่ต้องมี

> **บทนี้ไม่ต้องใช้ข้อมูลดิบ** — ใช้ไฟล์ `outputs/features_cache.npz` ที่อยู่ใน repo อยู่แล้ว รันได้เลย

---

## 5.1 ทำไมต้อง LSTM

ข้อมูลของเราเป็น**ลำดับเวลา** 20 จุดเรียงกัน โมเดลธรรมดา (เช่น Linear Regression
หรือ Neural Network ทั่วไป) มองข้อมูลเป็นแค่กองตัวเลข ไม่รู้ว่าอันไหนมาก่อนมาหลัง

**RNN (Recurrent Neural Network)** แก้ปัญหานี้ด้วยการอ่านข้อมูลทีละจุด
พร้อมกับพก "ความจำ" ติดตัวไปเรื่อย ๆ:

```
จุดที่ 1 → [ความจำ] → จุดที่ 2 → [ความจำ] → ... → จุดที่ 20 → คำตอบ
```

**LSTM (Long Short-Term Memory)** คือ RNN รุ่นปรับปรุงที่มี "ประตู" ควบคุมความจำ 3 บาน:

| ประตู | หน้าที่ |
|---|---|
| **Forget gate** | ตัดสินใจว่าจะลืมอะไรจากความจำเดิม |
| **Input gate** | ตัดสินใจว่าจะจำอะไรใหม่เข้าไป |
| **Output gate** | ตัดสินใจว่าจะเอาอะไรจากความจำมาใช้ตอบ |

ประตูพวกนี้แก้ปัญหา *vanishing gradient* ที่ทำให้ RNN ธรรมดาลืมข้อมูลเก่าเร็วเกินไป

ข่าวดีคือ PyTorch มี `nn.LSTM` ให้แล้ว เราไม่ต้องเขียนประตูเอง

In [ ]:
# ── ตั้งค่าให้ notebook มองเห็นโค้ดใน src/ ──
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ── หาฟอนต์ที่แสดงภาษาไทยได้ ไม่งั้นข้อความในกราฟจะกลายเป็นสี่เหลี่ยม ──
_installed = {f.name for f in fm.fontManager.ttflist}
for _f in ["Noto Sans Thai", "Leelawadee UI", "Tahoma", "TH Sarabun New", "Angsana New"]:
    if _f in _installed:
        plt.rcParams["font.family"] = _f
        plt.rcParams["axes.unicode_minus"] = False    # ฟอนต์ไทยมักไม่มีเครื่องหมายลบแบบ unicode
        print("ฟอนต์กราฟ:", _f)
        break
else:
    print("[หมายเหตุ] ไม่พบฟอนต์ไทย - ข้อความไทยในกราฟอาจแสดงเป็นสี่เหลี่ยม")
    print("           Windows/macOS มักมีอยู่แล้ว ส่วน Linux ลง: sudo apt install fonts-thai-tlwg")

print("project root:", ROOT)

In [ ]:
import torch
import torch.nn as nn

from src.paths import FEATURES_CACHE
from src.data_loader import compute_health_index
from src.dataset import split_dataset

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("device :", device)

In [ ]:
features = np.load(FEATURES_CACHE)["features"]
labels = compute_health_index(features, window=7)

WINDOW = 20
train_loader, val_loader, test_loader, scaler = split_dataset(
    features, labels, window_size=WINDOW, batch_size=32,
    shuffle_split=True, random_seed=42,
)

## 5.2 เขียนโมเดล

โครงสร้างที่เราจะสร้าง:

```
ข้อมูลเข้า (batch, 20, 56)
      ↓
   LSTM(128)          อ่านลำดับ เก็บความจำ 128 ค่า
      ↓
   LSTM(64)           กลั่นอีกชั้น
      ↓
 เอาเฉพาะจุดสุดท้าย   (batch, 64)
      ↓
   Linear(64 → 32) + ReLU
      ↓
   Linear(32 → 1)
      ↓
   Sigmoid            บีบให้อยู่ในช่วง 0-1
      ↓
 Health Index (batch,)
```

**ทำไมต้อง Sigmoid ปิดท้าย?** เพราะ Health Index อยู่ในช่วง 0–1 อยู่แล้ว
การบังคับให้ output อยู่ในช่วงนั้นช่วยให้โมเดลเรียนง่ายขึ้น

In [ ]:
class MyLSTM(nn.Module):
    def __init__(self, n_features=56, hidden1=128, hidden2=64):
        super().__init__()
        # batch_first=True หมายถึงข้อมูลเข้าเป็น (batch, seq, features)
        self.lstm1 = nn.LSTM(n_features, hidden1, batch_first=True)
        self.lstm2 = nn.LSTM(hidden1, hidden2, batch_first=True)
        self.fc1 = nn.Linear(hidden2, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        # x: (batch, 20, 56)
        x, _ = self.lstm1(x)        # -> (batch, 20, 128)
        x, _ = self.lstm2(x)        # -> (batch, 20, 64)
        x = x[:, -1, :]             # เอาเฉพาะ timestep สุดท้าย -> (batch, 64)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return torch.sigmoid(x).squeeze(-1)    # -> (batch,)


model = MyLSTM().to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nจำนวนพารามิเตอร์ที่ต้องเรียน: {n_params:,}")

### ⚠️ จุดที่พลาดกันบ่อย: `.squeeze(-1)`

สังเกตบรรทัดสุดท้ายของ `forward` — ถ้าไม่มี `.squeeze(-1)`
output จะมี shape `(batch, 1)` แต่คำตอบมี shape `(batch,)`

PyTorch จะไม่ error แต่จะ **broadcast** เป็นเมทริกซ์ `(batch, batch)`
ทำให้ loss ผิดทั้งหมดโดยไม่มีใครรู้ — เป็นบั๊กที่หาเจอยากมาก

ลองดูว่ามันเกิดอะไรขึ้นจริง ๆ

In [ ]:
pred_wrong = torch.rand(8, 1)    # ลืม squeeze
pred_right = torch.rand(8)       # squeeze แล้ว
target     = torch.rand(8)

loss_fn = nn.MSELoss()
print("shape ของ loss เมื่อลืม squeeze:", loss_fn(pred_wrong, target).shape, "<- ค่าเดียวเหมือนกัน แต่...")
print()
print("ปัญหาอยู่ที่การลบกันภายใน:")
print("  (8,1) - (8,)  ->", (pred_wrong - target).shape, " <- กลายเป็นเมทริกซ์ 8x8!")
print("  (8,)  - (8,)  ->", (pred_right - target).shape, " <- ถูกต้อง")

## 5.3 เทรน

องค์ประกอบของการเทรน:

- **Loss function** — `MSELoss` วัดว่าทำนายห่างจากคำตอบแค่ไหน (ยกกำลังสอง)
- **Optimizer** — `AdamW` ตัวปรับน้ำหนักตาม gradient
- **Learning rate** — ก้าวละเท่าไร ใหญ่ไปจะกระโดดข้าม เล็กไปจะช้า
- **Epoch** — หนึ่งรอบที่โมเดลเห็นข้อมูล train ครบทุกตัว

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()          # ล้าง gradient เก่า
        pred = model(xb)               # ทำนาย
        loss = loss_fn(pred, yb)       # วัดความผิดพลาด
        loss.backward()                # คำนวณ gradient
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # กัน gradient ระเบิด
        optimizer.step()               # ปรับน้ำหนัก

        total += loss.item() * len(xb)
    return total / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total, preds, targets = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        total += loss_fn(pred, yb).item() * len(xb)
        preds.append(pred.cpu().numpy())
        targets.append(yb.cpu().numpy())
    return total / len(loader.dataset), np.concatenate(preds), np.concatenate(targets)


print("ฟังก์ชันพร้อมแล้ว")

### Early stopping — ทำไมต้องมี

ถ้าเทรนไปเรื่อย ๆ โมเดลจะเริ่ม**ท่องจำ**ข้อมูล train แทนที่จะเรียนรู้รูปแบบ
เรียกว่า **overfitting** สังเกตได้จาก train loss ยังลดแต่ val loss เริ่มเพิ่ม

**Early stopping** = หยุดเมื่อ val loss ไม่ดีขึ้นติดกันหลาย epoch
และเก็บ checkpoint ตอนที่ val loss ต่ำที่สุดไว้

In [ ]:
import copy, time

EPOCHS = 60
PATIENCE = 15

model = MyLSTM().to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

history = {"train": [], "val": []}
best_val, best_state, patience_counter, best_epoch = float("inf"), None, 0, 0

t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tr = train_one_epoch(model, train_loader, optimizer, loss_fn)
    va, _, _ = evaluate(model, val_loader, loss_fn)
    history["train"].append(tr)
    history["val"].append(va)

    if va < best_val:
        best_val, best_epoch = va, epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        mark = " <- ดีที่สุด"
    else:
        patience_counter += 1
        mark = ""

    if epoch % 5 == 0 or mark:
        print(f"epoch {epoch:>3} | train {tr:.6f} | val {va:.6f}{mark}")

    if patience_counter >= PATIENCE:
        print(f"\nหยุดที่ epoch {epoch} - val loss ไม่ดีขึ้นติดกัน {PATIENCE} รอบ")
        break

model.load_state_dict(best_state)     # ย้อนกลับไปใช้ checkpoint ที่ดีที่สุด
print(f"\nใช้เวลา {time.time()-t0:.1f} วินาที | val loss ดีที่สุด {best_val:.6f} (epoch {best_epoch})")

## 5.4 อ่านกราฟ loss

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(history["train"], label="Train loss", color="#888", linewidth=1.8)
plt.plot(history["val"], label="Val loss", color="#d32f2f", linewidth=2)
plt.axvline(best_epoch - 1, color="#2e7d32", linestyle="--", linewidth=1.5,
            label=f"checkpoint ที่เลือก (epoch {best_epoch})")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss (log scale)")
plt.title("เส้นโค้งการเรียนรู้")
plt.legend()
plt.tight_layout()
plt.show()

**วิธีอ่านกราฟนี้:**

| สิ่งที่เห็น | ความหมาย |
|---|---|
| ทั้งสองเส้นลงพร้อมกัน | กำลังเรียนรู้ได้ดี |
| train ลง แต่ val เริ่มขึ้น | เริ่ม overfit — early stopping ควรตัดตรงนี้ |
| ทั้งสองเส้นนิ่งตั้งแต่ต้น | learning rate น้อยไป หรือโมเดลเล็กไป |
| เส้นกระโดดขึ้นลงรุนแรง | learning rate มากไป |

ใช้ log scale เพราะ loss ลดลงหลายเท่าตัว ถ้าใช้สเกลปกติจะเห็นแต่ช่วงต้น

## 5.5 วัดผลบน Test set

จุดสำคัญ: **นี่เป็นครั้งแรกและครั้งเดียวที่เราแตะ Test set**

In [ ]:
from scipy.stats import pearsonr

test_loss, preds, targets = evaluate(model, test_loader, loss_fn)

rmse = float(np.sqrt(np.mean((preds - targets) ** 2)))
mae  = float(np.mean(np.abs(preds - targets)))
r, _ = pearsonr(targets, preds)

print(f"RMSE      : {rmse:.4f}")
print(f"MAE       : {mae:.4f}")
print(f"Pearson r : {r:.4f}")

In [ ]:
# เรียงตามค่าจริงเพื่อให้ดูง่าย (test set ถูกสุ่มลำดับไว้)
order = np.argsort(-targets)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(targets[order], label="ค่าจริง", color="#222", linewidth=2)
axes[0].plot(preds[order], label="ทำนาย", color="#d32f2f", linewidth=1.2, alpha=0.85)
axes[0].set_xlabel("ตัวอย่างใน test set (เรียงตามค่าจริง)")
axes[0].set_ylabel("Health Index")
axes[0].set_title("ค่าจริง vs ค่าที่ทำนาย")
axes[0].legend()

axes[1].scatter(targets, preds, s=14, alpha=0.6, color="#1976d2")
lims = [targets.min(), targets.max()]
axes[1].plot(lims, lims, "k--", linewidth=1.2, label="ทำนายสมบูรณ์แบบ")
axes[1].set_xlabel("ค่าจริง")
axes[1].set_ylabel("ค่าที่ทำนาย")
axes[1].set_title("ยิ่งจุดเกาะเส้นประยิ่งแม่น")
axes[1].legend()

plt.tight_layout()
plt.show()

**อย่าเพิ่งดีใจกับตัวเลข RMSE ที่ดูน้อย** — บทที่ 7 จะแสดงให้เห็นว่า
ทำไมตัวเลขนี้ยังบอกอะไรไม่ได้จนกว่าจะเทียบกับ baseline

## 🔧 ลองแก้ดู — ทดลองกับโมเดล


1. เปลี่ยน `lr=5e-4` เป็น `5e-2` (ใหญ่ขึ้น 100 เท่า) แล้วเทรนใหม่ —
   กราฟ loss หน้าตาเป็นอย่างไร? แล้วลอง `5e-6` ล่ะ?
2. ลดขนาดโมเดลเหลือ `MyLSTM(hidden1=16, hidden2=8)` — ผลแย่ลงมากไหม?
   บอกอะไรเราเกี่ยวกับความยากของปัญหานี้?
3. ปิด early stopping โดยตั้ง `PATIENCE = 999` แล้วเทรนครบ 60 epoch —
   val loss แย่ลงตอนท้ายไหม?
4. ลองเปลี่ยนไปใช้ `x.mean(dim=1)` แทน `x[:, -1, :]` ในบรรทัดที่เลือก timestep
   (เฉลี่ยทุก timestep แทนการเอาตัวสุดท้าย) — ผลต่างกันอย่างไร?

## ❓ เช็คความเข้าใจ

**1. ทำไมต้องเก็บ checkpoint ที่ val loss ต่ำสุด แทนที่จะใช้โมเดลตอนจบการเทรน?**

<details>
<summary>ดูเฉลย</summary>

เพราะโมเดลตอนจบอาจ overfit ไปแล้ว val loss ที่ต่ำที่สุดคือจุดที่โมเดลสมดุลที่สุด ระหว่างการเรียนรู้รูปแบบกับการท่องจำ การใช้ checkpoint นั้นจึงให้ผลบนข้อมูลใหม่ดีกว่า

</details>

**2. `optimizer.zero_grad()` ทำอะไร ถ้าลืมใส่จะเกิดอะไรขึ้น?**

<details>
<summary>ดูเฉลย</summary>

ล้าง gradient ที่ค้างจากรอบก่อน เพราะ PyTorch สะสม gradient ทับกันไปเรื่อย ๆ ถ้าลืม gradient จากทุก batch จะบวกกันหมด ทำให้การปรับน้ำหนักผิดพลาดรุนแรง และโมเดลจะไม่ลู่เข้า

</details>

**3. ทำไมเราถึงใช้ Test set แค่ครั้งเดียวตอนจบ?**

<details>
<summary>ดูเฉลย</summary>

เพราะทุกครั้งที่ดูผล Test แล้วกลับไปปรับอะไร เท่ากับเราใช้ข้อมูล Test ในการตัดสินใจ ตัวเลขที่ได้ก็จะไม่ใช่การประเมินบนข้อมูลใหม่จริง ๆ อีกต่อไป — นี่คือเหตุผลที่ต้องมี Validation set แยกไว้สำหรับการตัดสินใจระหว่างทาง

</details>

---

## สรุปบทนี้

- LSTM อ่านข้อมูลทีละจุดพร้อมพกความจำ เหมาะกับข้อมูลลำดับเวลา
- ระวัง shape ของ output ให้ตรงกับ label ไม่งั้น broadcast จะทำให้ loss ผิดเงียบ ๆ
- Early stopping กัน overfit และเก็บ checkpoint ที่ดีที่สุดไว้
- อ่านกราฟ loss เป็น จะบอกได้ว่าปัญหาอยู่ที่ learning rate ขนาดโมเดล หรือ overfit

[← บทที่ 4](04_data_preparation.ipynb) &nbsp;·&nbsp; [สารบัญ](README.md) &nbsp;·&nbsp; **[บทที่ 6 — เทียบ 4 สถาปัตยกรรม →](06_comparing_architectures.ipynb)**